In [ ]:
# pipeline.py 抜粋（前処理 clean_df の直後）

from .embeddings import encode_texts_with_cache
from .dedup import deduplicate_headlines

cache_path = cfg.resolve_path(cfg.embedding.headline_cache_path)
assert cache_path is not None

# 1) headline_clean を embedding 化（キャッシュ再利用）
emb_meta, all_embeddings = encode_texts_with_cache(
    ids=clean_df["news_id"],
    texts=clean_df["headline_clean"],
    cache_path=cache_path,
    model_name=cfg.embedding.model_name,
    batch_size=cfg.embedding.batch_size,
    normalize_embeddings=cfg.embedding.normalize_embeddings,
)

# 2) 重複抑制
dedup_df, dedup_log = deduplicate_headlines(
    clean_df,
    embeddings=all_embeddings,
    similarity_threshold=cfg.dedup.similarity_threshold,  # 既定 0.92
    window_hours=cfg.dedup.window_hours,                  # 既定 24h
)


In [ ]:
# pipeline.py 抜粋（重複除去の次）

def _select_fit_sample(df: pd.DataFrame, cfg: PipelineConfig) -> pd.DataFrame:
    out = df.copy()
    out["timestamp"] = pd.to_datetime(out["timestamp"], utc=True, errors="coerce")

    fit_start = cfg.topic_model.fit_start
    fit_end = cfg.topic_model.fit_end

    if fit_start:
        out = out[out["timestamp"] >= pd.to_datetime(fit_start, utc=True)]
    if fit_end:
        out = out[out["timestamp"] <= pd.to_datetime(fit_end, utc=True)]

    if out.empty:
        raise ValueError("BERTopic学習期間に該当するニュースがありません。")
    return out.reset_index(drop=True)

# dedup済みニュースに対応する embedding を再取得
clean_pos = clean_df.reset_index()[["index", "news_id"]]
dedup_indexer = (
    dedup_df[["news_id"]]
    .merge(clean_pos, on="news_id", how="left")["index"]
    .to_numpy(dtype=int)
)
dedup_emb = all_embeddings[dedup_indexer]

# fit対象期間を抽出し、対応embeddingを取得
fit_df = _select_fit_sample(dedup_df, cfg)
dedup_pos = dedup_df.reset_index()[["index", "news_id"]]
fit_idx = (
    fit_df[["news_id"]]
    .merge(dedup_pos, on="news_id", how="left")["index"]
    .to_numpy(dtype=int)
)
fit_emb = dedup_emb[fit_idx]

# BERTopic fit -> full transform
topic_model, fit_tables = fit_topic_model(fit_df, fit_emb, cfg.topic_model)
topic_assignments = transform_topics(dedup_df, topic_model, dedup_emb)
topic_tables = build_topic_tables(topic_assignments, topic_model, cfg.topic_model.top_n_headlines)


In [ ]:
# topics.py 抜粋（BERTopic本体）

def _build_topic_model(settings: TopicModelSettings):
    from bertopic import BERTopic
    from hdbscan import HDBSCAN
    from umap import UMAP

    umap_model = UMAP(
        n_neighbors=settings.umap_n_neighbors,
        n_components=settings.umap_n_components,
        min_dist=settings.umap_min_dist,
        metric=settings.umap_metric,
        random_state=settings.random_state,
    )
    hdbscan_model = HDBSCAN(
        min_cluster_size=settings.hdbscan_min_cluster_size,
        metric=settings.hdbscan_metric,
        cluster_selection_method=settings.hdbscan_cluster_selection_method,
        prediction_data=True,
    )

    vectorizer_model = build_japanese_vectorizer(settings.tokenizer_stopwords)

    return BERTopic(
        embedding_model=None,          # 事前計算embeddingを使う
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        top_n_words=settings.top_n_words,
        calculate_probabilities=settings.calculate_probabilities,
        language=settings.language,
        verbose=False,
    )

def fit_topic_model(train_df: pd.DataFrame, embeddings: np.ndarray, settings: TopicModelSettings):
    docs = train_df["headline_clean"].fillna("").astype(str).tolist()
    if len(docs) != len(embeddings):
        raise ValueError("train_df と embeddings の行数が一致しません。")

    topic_model = _build_topic_model(settings)
    topics, probs = topic_model.fit_transform(docs, embeddings=embeddings)

    assignments = _build_assignments(train_df, topics, probs)
    topic_summary = _build_topic_summary(assignments, topic_model, settings.top_n_headlines)

    outlier_ratio = float((assignments["topic_id"] == -1).mean()) if len(assignments) > 0 else np.nan
    outlier_stats = pd.DataFrame([{
        "n_docs": int(len(assignments)),
        "n_outliers": int((assignments["topic_id"] == -1).sum()),
        "outlier_ratio": outlier_ratio
    }])

    return topic_model, {
        "train_assignments": assignments,
        "topic_summary": topic_summary,
        "outlier_stats": outlier_stats,
    }

def transform_topics(full_df: pd.DataFrame, topic_model, embeddings: np.ndarray) -> pd.DataFrame:
    docs = full_df["headline_clean"].fillna("").astype(str).tolist()
    topics, probs = topic_model.transform(docs, embeddings=embeddings)
    return _build_assignments(full_df, topics, probs)
